<a href="https://colab.research.google.com/github/aishwaryaaramanathan/DL_Lab_241501011/blob/main/dl_exp_8.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, Subset
import matplotlib.pyplot as plt

In [ ]:
# Load dataset
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
latent_dim = 64

transform = transforms.Compose([
    transforms.CenterCrop(178),
    transforms.Resize((64, 64)),
    transforms.ToTensor(),
    transforms.Normalize((0.5,)*3, (0.5,)*3)  ])

dataset = datasets.CelebA(
    root='./data', split='train', download=True, transform=transform )
loader = DataLoader(
    Subset(dataset, range(500)), batch_size=64, shuffle=True )

In [4]:
# Define VAE
class VAE(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Conv2d(3, 32, 4, 2, 1), nn.ReLU(),
            nn.Conv2d(32, 64, 4, 2, 1), nn.ReLU(),
            nn.Flatten()
        )
        self.mu = nn.Linear(64*16*16, latent_dim)
        self.logvar = nn.Linear(64*16*16, latent_dim)

        self.fc = nn.Linear(latent_dim, 64*16*16)
        self.decoder = nn.Sequential(
            nn.ConvTranspose2d(64, 32, 4, 2, 1), nn.ReLU(),
            nn.ConvTranspose2d(32, 3, 4, 2, 1), nn.Tanh()
        )

    def forward(self, x):
        h = self.encoder(x)
        mu, logvar = self.mu(h), self.logvar(h)
        z = mu + torch.randn_like(mu) * torch.exp(0.5 * logvar)
        out = self.fc(z).view(-1, 64, 16, 16)
        return self.decoder(out), mu, logvar


In [5]:
# Train VAE
model = VAE().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

for epoch in range(5):
    total_loss = 0
    model.train()

    for images, _ in loader:
        images = images.to(device)
        recon, mu, logvar = model(images)

        recon_loss = F.mse_loss(recon, images, reduction='sum')
        kl_loss = -0.5 * torch.sum(
            1 + logvar - mu.pow(2) - logvar.exp()
        )
        loss = recon_loss + kl_loss

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    print(f"Epoch {epoch+1}, Loss: {total_loss/len(loader.dataset):.4f}")

NameError: name 'loader' is not defined

In [ ]:
# Generate new images
model.eval()
with torch.no_grad():
    images = model.decoder(
        model.fc(torch.randn(16, latent_dim).to(device))
          .view(-1, 64, 16, 16)
    ).cpu()
    images = images * 0.5 + 0.5

grid = torchvision.utils.make_grid(images, nrow=4)
plt.imshow(grid.permute(1, 2, 0).clamp(0, 1))
plt.axis('off')
plt.title("Generated Images")
plt.show()